# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rafiaburdibaloch-ui/flyrank-rafia/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

In [33]:
import pandas as pd
import numpy as np

In [34]:
import duckdb

con = duckdb.connect()
print("DuckDB connected successfully")

DuckDB connected successfully


In [35]:
import os

path = "data/raw/content_refresh_anonymized.csv"

print("File exists:", os.path.exists(path))

File exists: False


In [36]:
import os

for root, dirs, files in os.walk("."):
    level = root.replace(".", "").count(os.sep)
    if level <= 3:
        print(root, files[:10])

. []
./.config ['config_sentinel', 'gce', '.last_opt_in_prompt.yaml', '.last_survey_prompt.yaml', 'hidden_gcloud_config_universe_descriptor_data_cache_configs.db', 'active_config', 'default_configs.db', '.last_update_check.json']
./.config/logs []
./.config/logs/2026.09.16 ['13.26.04.283691.log', '13.26.17.091876.log', '13.26.29.139725.log', '13.26.30.459687.log', '13.26.15.129583.log', '13.25.37.877256.log']
./.config/configurations ['config_default']
./flyrank-rafia ['01_first_look_and_discovery.ipynb', 'LICENSE', 'CLAUDE.md', 'SETUP.md', 'AGENTS.md', 'README.md', 'requirements.txt', 'DATA_USE.md', '.gitignore', 'GUIDE.md']
./flyrank-rafia/work ['README.md', 'capstone_report_template.md']
./flyrank-rafia/work/notebooks ['w01_research_question.ipynb', 'w03_data_contract.ipynb', 'w03_feature_leakage_check.ipynb', 'w04_signal_audit.ipynb', 'w07_action_playbook.ipynb', 'w06_validation_audit.ipynb', 'w05_model.ipynb', 'w02_ml_task_framing.ipynb', 'w04_baseline_score.ipynb', 'capstone.ipyn

In [37]:
!git clone https://github.com/rafiaburdibaloch-ui/flyrank-rafia.git

fatal: destination path 'flyrank-rafia' already exists and is not an empty directory.


In [38]:
import os

for root, dirs, files in os.walk("flyrank-rafia"):
    level = root.replace("flyrank-rafia", "").count(os.sep)

    if level <= 3:
        print(root)
        for file in files[:15]:
            print("   ", file)

flyrank-rafia
    01_first_look_and_discovery.ipynb
    LICENSE
    CLAUDE.md
    SETUP.md
    AGENTS.md
    README.md
    requirements.txt
    DATA_USE.md
    .gitignore
    GUIDE.md
flyrank-rafia/work
    README.md
    capstone_report_template.md
flyrank-rafia/work/notebooks
    w01_research_question.ipynb
    w03_data_contract.ipynb
    w03_feature_leakage_check.ipynb
    w04_signal_audit.ipynb
    w07_action_playbook.ipynb
    w06_validation_audit.ipynb
    w05_model.ipynb
    w02_ml_task_framing.ipynb
    w04_baseline_score.ipynb
    capstone.ipynb
flyrank-rafia/.github
flyrank-rafia/.github/workflows
    personalize.yml
    smoke-test.yml
    data-path-smoke.yml
flyrank-rafia/outputs
    refresh_queue_sample.csv
    model_report.md
flyrank-rafia/outputs/charts
    action_mix.svg
    top_feature_importance.svg
    trend_distribution.svg
    confidence_mix.svg
    top_reason_codes.svg
flyrank-rafia/.git
    HEAD
    config
    index
    packed-refs
    description
flyrank-rafia/.gi

In [39]:
from pathlib import Path

print(Path("flyrank-rafia/skills/flyrank/flyrank-data/SKILL.md").read_text())

---
name: flyrank-data
description: The FlyRank internship datasets — the 30k-row starter CSV and its gotchas, the ~79M-row warehouse release tables and grains, panel warnings, access, and iteration rules. Load for EVERY task that touches the data. (Project-specific: delete this folder when reusing the skill library elsewhere.)
---

# FlyRank internship data

Two datasets. The small one ships in this repo; the big one is hosted and gated.

## 1. Starter dataset (in this repo)

`data/raw/content_refresh_anonymized.csv` — 30,000 rows × 44 columns, one row per pseudonymized
content item, 32 clients, trailing-90-day metrics. Full column reference: `docs/data-dictionary.md`
(keep it open). The gotchas that cause 90% of mistakes:

- **Rate columns are ×100 percentages**: `ctr = 0.76` means 0.76%, not 76%. Applies to ctr,
  engagement_rate, scroll_rate, ai_traffic_pct, trend_pct.
- **`avg_position = 0` means "no data"**, not rank zero (1,205 rows).
- **`scroll_rate` and `ai_traffic_pct` can e

In [40]:
id="v9k3zq"
import pandas as pd

file_path = "flyrank-rafia/data/raw/content_refresh_anonymized.csv"

df = pd.read_csv(file_path)

print(df.shape)

(30000, 44)


In [41]:
print(df.columns.tolist())

['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']


In [42]:
age_check = (
    df.groupby("age_tier")
    .agg(
        n=("content_id", "count"),
        avg_age_days=("content_age_days", "mean")
    )
    .reset_index()
)

age_check

,age_tier,n,avg_age_days
0,181-365,11368,281.140482
1,31-90,492,90.000000
2,365+,6360,462.690566
3,91-180,11780,127.507385


In [43]:
[c for c in df.columns if "refresh" in c.lower() or "flag" in c.lower()]

[]

In [44]:
from pathlib import Path

dictionary = Path("flyrank-rafia/docs/data-dictionary.md").read_text()

print(dictionary[:5000])

# Data Dictionary — `content_refresh_anonymized.csv`

One row per content item (page): **30,000 rows × 44 columns**, covering **32 pseudonymized
clients**. All metrics are aggregated over a trailing 90-day window ending at export time.
Keep this file open while you work.

## Read this first — the three rules that prevent 90% of mistakes

1. **Rate columns are ×100 percentages.** `ctr = 0.76` means **0.76%**, not 76%. Applies to
   `ctr`, `engagement_rate`, `scroll_rate`, `ai_traffic_pct`, `trend_pct`.
2. **The label comes from `trend_direction`.** The pipeline defines
   `is_declining_label = (trend_direction == "down")`, so `trend_direction` and `trend_pct`
   must **never** be model features — that's the leakage notebook 02 demonstrates.
3. **IDs are for grouping only.** `content_id` / `client_id` are pseudonyms: use them for
   joins and grouped train/test splits, never as features.

## Identifiers

| Column | Type | Meaning | Notes |
|---|---|---|---|
| `content_id` | text | Pseudo

In [45]:
from pathlib import Path

print(Path("flyrank-rafia").exists())

from pathlib import Path

skill = Path("flyrank-rafia/skills/building-baselines/SKILL.md").read_text()

print(skill)

True
---
name: building-baselines
description: Builds the transparent rule-based baseline every model must beat — a hand-written score with reason codes, ranked output, and precision@K evaluation. Use before training any model, or when someone reports model results with nothing to compare against.
---

# Building baselines

A model without a baseline is a number without a meaning. The baseline is a rule a human can
read — and its job is to be honestly beatable.

## Build it in this order

**1. Say the rule in plain words first.** "A page is worth reviewing if it used to get traffic,
it's getting old, and its position is slipping." If you can't say it, you can't code it.

**2. Code it as a transparent score.** Multiply/add simple conditions; no fitted weights:

```python
stale   = (df["days_since_update"] >= 180).astype(int)
visible = (df["impressions"] >= 500).astype(int)
df["score"] = stale * visible * df["impressions"]     # readable on purpose
```

**3. Attach reason codes.** Every 

In [46]:
import json
from pathlib import Path

nb = json.loads(
    Path("flyrank-rafia/work/notebooks/w04_signal_audit.ipynb").read_text()
)

for cell in nb["cells"]:
    text = "".join(cell.get("source", []))

    if any(word in text.lower() for word in ["refresh", "stale", "flag", "quick-win"]):
        print("=" * 60)
        print(text)

# ML-06 — Signal Audit: Do the Flags Hold?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rafiaburdibaloch-ui/flyrank-rafia/blob/main/work/notebooks/w04_signal_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.
## 3. The flag-linked test

*Pick a signal one of FlyRank's real flags relies on. Does the data support the rule's assumption?*


In [47]:
for i, cell in enumerate(nb["cells"]):
    text = "".join(cell.get("source", []))

    if "The flag-linked test" in text:
        print("Section 3 starts at cell:", i)
        print("\n--- SECTION 3 AND NEXT CELLS ---\n")

        for j in range(i, min(i + 8, len(nb["cells"]))):
            print(f"\n### CELL {j} ###")
            print("".join(nb["cells"][j].get("source", [])))

Section 3 starts at cell: 5

--- SECTION 3 AND NEXT CELLS ---


### CELL 5 ###
## 3. The flag-linked test

*Pick a signal one of FlyRank's real flags relies on. Does the data support the rule's assumption?*

### CELL 6 ###
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


### CELL 7 ###
## 4. What this means in practice

*Two or three sentences: what a content team should take from this.*

### CELL 8 ###
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


### CELL 9 ###
## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, di

In [48]:
df["days_since_last_update"].describe()

,days_since_last_update
count,30000.000000
mean,46.098300
std,42.078709
min,1.000000
25%,20.000000
50%,20.000000
75%,104.000000
max,373.000000


Signal 1 — Freshness: OPPOSITE

Pages that have gone longer without an update did not show weaker performance in this dataset. The 181+ day group had the highest average CTR (3.69%) and best average position (11.33), although it was small (n=174). I will not use freshness alone as a reason to prioritize a page.

In [49]:
freshness_check = (
    df.assign(
        update_bucket=pd.cut(
            df["days_since_last_update"],
            bins=[0, 30, 90, 180, np.inf],
            labels=["0-30", "31-90", "91-180", "181+"]
        )
    )
    .groupby("update_bucket", observed=False)
    .agg(
        n=("content_id", "count"),
        avg_ctr=("ctr", "mean"),
        avg_position=("avg_position", "mean")
    )
    .reset_index()
)

freshness_check

,update_bucket,n,avg_ctr,avg_position
0,0-30,20480,0.609021,15.685166
1,31-90,175,0.117543,16.538286
2,91-180,9171,0.238367,17.901461
3,181+,174,3.693276,11.325862


Signal 2 — CTR vs position: CONFIRMED

CTR decreases as average search position gets worse. Pages in positions 1–3 had an average CTR of 2.71%, compared with 0.15% for pages at 51+. This supports using CTR relative to search position when identifying pages that may need attention.

In [50]:
df["avg_position"].describe()
position_ctr_check = (
    df.assign(
        position_bucket=pd.cut(
            df["avg_position"],
            bins=[0, 3, 10, 20, 50, np.inf],
            labels=["1-3", "4-10", "11-20", "21-50", "51+"]
        )
    )
    .groupby("position_bucket", observed=False)
    .agg(
        n=("content_id", "count"),
        avg_ctr=("ctr", "mean")
    )
    .reset_index()
)

position_ctr_check

,position_bucket,n,avg_ctr
0,1-3,1141,2.714303
1,4-10,11842,0.651045
2,11-20,7273,0.323443
3,21-50,7225,0.222345
4,51+,1314,0.150784


In [51]:
import os

for root, dirs, files in os.walk("flyrank-rafia"):
    for file in files:
        if file.endswith((".py", ".md", ".ipynb", ".csv")):
            path = os.path.join(root, file)
            try:
                text = open(path, encoding="utf-8").read().lower()
                if any(word in text for word in ["quick-win", "quick win", "refresh flag", "refresh_flag", "ctr-fix", "ctr fix"]):
                    print(path)
            except:
                pass

flyrank-rafia/notebooks/02_your_first_readable_model.ipynb
flyrank-rafia/skills/flyrank/flyrank-context/SKILL.md
flyrank-rafia/docs/ml-intern-dataset-and-lane-guide.md


In [52]:
from pathlib import Path

path = "flyrank-rafia/skills/flyrank/flyrank-context/SKILL.md"
text = Path(path).read_text(encoding="utf-8")

print(text)

---
name: flyrank-context
description: FlyRank-specific background — what the company does, the search problem, and the hand-written flags story that the ML internship work builds on. Load for context when an assignment references FlyRank's product, flags, or the "beat the rule" framing. (Project-specific: delete this folder when reusing the skill library elsewhere.)
---

# FlyRank context

**What FlyRank is.** FlyRank builds content as infrastructure: it researches, writes, publishes
content straight into a client's website, then watches search data and optimizes — the full
cycle, run by algorithms and AI rather than people doing it by hand.

**The problem this internship works on.** Content that gets found in Google search ranks, then
quietly decays: rankings slip, clicks drop, and most teams notice too late. The valuable
decision is: *out of thousands of pages, which one should a human fix FIRST?* That is a ranking
problem on messy, real search data — exactly where a learned model c

In [53]:
from pathlib import Path

path = "flyrank-rafia/docs/ml-intern-dataset-and-lane-guide.md"
text = Path(path).read_text(encoding="utf-8")

for i, line in enumerate(text.splitlines(), 1):
    if any(word in line.lower() for word in ["quick-win", "quick win", "refresh", "ctr", "flag", "needs-attention"]):
        print(f"{i}: {line}")

12: > This starter repo ships only the small anonymized starter dataset (`data/raw/content_refresh_anonymized.csv`).
25: - its verified committed outputs (`outputs/model_report.md`, `outputs/refresh_queue_sample.csv`, `outputs/charts/`); `outputs/model_results.json` is regenerated when you run the pipeline;
26: - the small anonymized starter dataset (`data/raw/content_refresh_anonymized.csv`).
54: - Treat product scores and flags as useful background, not as the truth.
86: | Refresh / Content Opportunity Scoring | Core lane | starter playground plus warehouse support | Ranked review queue with scores, actions, and reason codes |
88: | CTR / Engagement Opportunity Scoring | Core lane | warehouse release or starter dataset | Ranked opportunity score; position/volume adjusted |
146: - The CTR and engagement lanes have enough direct signal, but you will still need minimum-volume filters (rules like "only look at pages with at least N impressions") to keep noise out.
178: | Observed signals

Signal 2 — FlyRank low_ctr_visible_page flag: CONFIRMED

The documented FlyRank flag identifies pages with at least 500 impressions, position 1–20, and CTR below 0.5%. In this dataset, 9,759 pages meet the flag conditions and have an average CTR of 0.185%, compared with 0.668% for the 20,241 pages that do not. This supports using visible low-CTR pages as a review signal.

In [54]:
low_ctr_visible = (
    (df["impressions_90d"] >= 500)
    & (df["avg_position"] > 0)
    & (df["avg_position"] <= 20)
    & (df["ctr"] < 0.5)
)

flag_check = (
    df.assign(low_ctr_visible_page=low_ctr_visible)
    .groupby("low_ctr_visible_page")
    .agg(
        n=("content_id", "count"),
        avg_ctr=("ctr", "mean"),
        avg_position=("avg_position", "mean")
    )
    .reset_index()
)

flag_check

,low_ctr_visible_page,n,avg_ctr,avg_position
0,False,20241,0.667691,19.649642
1,True,9759,0.185190,9.482836


In [55]:
df[[
    "impressions_90d",
    "avg_position",
    "ctr"
]].describe()

,impressions_90d,avg_position,ctr
count,30000.000000,30000.00000,30000.000000
mean,5200.366300,16.34238,0.510733
std,16838.019547,15.21679,3.279162
min,1.000000,0.00000,0.000000
25%,81.000000,6.20000,0.000000
50%,731.000000,10.80000,0.070000
75%,3615.250000,22.30000,0.290000
max,517715.000000,245.00000,100.000000


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [56]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
queue = df.copy()

queue["eligible"] = (
    (queue["impressions_90d"] >= 500)
    & (queue["avg_position"] > 0)
    & (queue["avg_position"] <= 20)
)

queue["score"] = 0.0
queue["reason_code"] = "not_eligible"
queue["action"] = "monitor"

eligible = queue["eligible"]

queue.loc[eligible, "score"] = (
    (0.5 - queue.loc[eligible, "ctr"]).clip(lower=0) * 2
    + (20 - queue.loc[eligible, "avg_position"]) / 20
)

queue.loc[
    eligible & (queue["ctr"] < 0.5),
    "reason_code"
] = "low_ctr_visible_page"

queue.loc[
    eligible & (queue["ctr"] < 0.5),
    "action"
] = "review_ctr"

queue = queue.sort_values(
    ["score", "impressions_90d"],
    ascending=[False, False]
).reset_index(drop=True)

queue[[
    "content_id",
    "score",
    "action",
    "reason_code",
    "impressions_90d",
    "avg_position",
    "ctr"
]].head(20)

,content_id,score,action,reason_code,impressions_90d,avg_position,ctr
0,content_d0351144852b,1.970,review_ctr,low_ctr_visible_page,884,0.6,0.0
1,content_7e97f31ba9bb,1.965,review_ctr,low_ctr_visible_page,603,0.7,0.0
2,content_c90bfc85694f,1.945,review_ctr,low_ctr_visible_page,3047,1.1,0.0
3,content_1e57b71527e4,1.940,review_ctr,low_ctr_visible_page,1213,1.2,0.0
4,content_4bb993e9270e,1.935,review_ctr,low_ctr_visible_page,1986,1.3,0.0
5,content_81e0dfd4c3cc,1.935,review_ctr,low_ctr_visible_page,720,1.3,0.0
6,content_eb3b2c3bbc34,1.935,review_ctr,low_ctr_visible_page,580,1.3,0.0
7,content_134631e65b9e,1.925,review_ctr,low_ctr_visible_page,4417,1.5,0.0
8,content_1d2233dc3323,1.925,review_ctr,low_ctr_visible_page,1463,1.5,0.0
9,content_97a08f7b106c,1.925,review_ctr,low_ctr_visible_page,1108,1.5,0.0


In [57]:
from pathlib import Path

output_dir = Path("flyrank-rafia/work/outputs")
output_dir.mkdir(parents=True, exist_ok=True)

output_path = output_dir / "baseline_action_score.csv"

queue[[
    "content_id",
    "score",
    "action",
    "reason_code",
    "impressions_90d",
    "avg_position",
    "ctr"
]].to_csv(output_path, index=False)

print(f"Saved: {output_path}")
print(f"Rows written: {len(queue)}")

Saved: flyrank-rafia/work/outputs/baseline_action_score.csv
Rows written: 30000


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

### Top-20 review

All top-20 pages are assigned **review_ctr** with reason code **low_ctr_visible_page** because they have at least 500 impressions, average position between 1 and 20, and CTR below 0.5%.

* **content_d0351144852b** — Review CTR; 884 impressions, position 0.6, CTR 0.0%. High confidence because it has strong visibility but no recorded clicks. It could be wrong if the CTR data is sparse or tracking is incomplete.
* **content_7e97f31ba9bb** — Review CTR; 603 impressions, position 0.7, CTR 0.0%. Strong position and sufficient volume support the review. It could be wrong if the measured clicks are incomplete.
* **content_c90bfc85694f** — Review CTR; 3,047 impressions, position 1.1, CTR 0.0%. Very strong visibility with no clicks. It could be wrong if tracking or attribution is unreliable.
* **content_1e57b71527e4** — Review CTR; 1,213 impressions, position 1.2, CTR 0.0%. Strong position and volume make this a clear review candidate. It could be wrong if the zero CTR is a measurement issue.
* **content_4bb993e9270e** — Review CTR; 1,986 impressions, position 1.3, CTR 0.0%. Good visibility with no recorded clicks. It could be wrong if click data is incomplete.
* **content_81e0dfd4c3cc** — Review CTR; 720 impressions, position 1.3, CTR 0.0%. Meets the volume and visibility conditions. It could be wrong if the observed CTR is unreliable.
* **content_eb3b2c3bbc34** — Review CTR; 580 impressions, position 1.3, CTR 0.0%. Sufficient impressions and strong position support review. It could be wrong if the sample is not representative.
* **content_134631e65b9e** — Review CTR; 4,417 impressions, position 1.5, CTR 0.0%. High visibility makes the zero CTR notable. It could be wrong if tracking is incomplete.
* **content_1d2233dc3323** — Review CTR; 1,463 impressions, position 1.5, CTR 0.0%. Strong position and sufficient volume support review. It could be wrong if clicks were under-recorded.
* **content_97a08f7b106c** — Review CTR; 1,108 impressions, position 1.5, CTR 0.0%. Good visibility with no recorded clicks. It could be wrong if the CTR measurement is incomplete.
* **content_fb4884e34467** — Review CTR; 580 impressions, position 1.5, CTR 0.0%. Meets the minimum-volume and position conditions. It could be wrong if the sample is too limited.
* **content_7a6cb4857fe6** — Review CTR; 789 impressions, position 1.6, CTR 0.0%. Strong position and enough impressions support review. It could be wrong if click tracking is incomplete.
* **content_6e6694b67eb6** — Review CTR; 1,969 impressions, position 1.7, CTR 0.0%. Good visibility with zero recorded CTR. It could be wrong if the observed data is incomplete.
* **content_35d0f98ef4dc** — Review CTR; 2,146 impressions, position 1.8, CTR 0.0%. Strong visibility makes this a useful review candidate. It could be wrong if clicks were not fully captured.
* **content_eb72d6efc172** — Review CTR; 630 impressions, position 1.8, CTR 0.0%. Meets the rule conditions clearly. It could be wrong if the CTR is affected by measurement issues.
* **content_53d353c3db0f** — Review CTR; 1,518 impressions, position 1.9, CTR 0.0%. Sufficient volume and strong position support review. It could be wrong if click data is incomplete.
* **content_c7b87d5dd8a0** — Review CTR; 1,418 impressions, position 1.9, CTR 0.0%. Strong search visibility with no recorded clicks. It could be wrong if CTR tracking is unreliable.
* **content_8cbda3f608d0** — Review CTR; 866 impressions, position 1.9, CTR 0.0%. Meets the volume and position conditions. It could be wrong if the observed zero CTR is not representative.
* **content_b5964a1a276f** — Review CTR; 1,872 impressions, position 2.0, CTR 0.0%. Good visibility and sufficient volume support review. It could be wrong if click measurement is incomplete.
* **content_0c8cb6bad4a7** — Review CTR; 502 impressions, position 2.0, CTR 0.0%. Just above the minimum volume threshold, so it is a weaker pick than higher-volume rows. It could be wrong if the smaller sample is unstable.

In [58]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
top20 = queue.head(20)[[
    "content_id",
    "score",
    "action",
    "reason_code",
    "impressions_90d",
    "avg_position",
    "ctr"
]]

top20

,content_id,score,action,reason_code,impressions_90d,avg_position,ctr
0,content_d0351144852b,1.970,review_ctr,low_ctr_visible_page,884,0.6,0.0
1,content_7e97f31ba9bb,1.965,review_ctr,low_ctr_visible_page,603,0.7,0.0
2,content_c90bfc85694f,1.945,review_ctr,low_ctr_visible_page,3047,1.1,0.0
3,content_1e57b71527e4,1.940,review_ctr,low_ctr_visible_page,1213,1.2,0.0
4,content_4bb993e9270e,1.935,review_ctr,low_ctr_visible_page,1986,1.3,0.0
5,content_81e0dfd4c3cc,1.935,review_ctr,low_ctr_visible_page,720,1.3,0.0
6,content_eb3b2c3bbc34,1.935,review_ctr,low_ctr_visible_page,580,1.3,0.0
7,content_134631e65b9e,1.925,review_ctr,low_ctr_visible_page,4417,1.5,0.0
8,content_1d2233dc3323,1.925,review_ctr,low_ctr_visible_page,1463,1.5,0.0
9,content_97a08f7b106c,1.925,review_ctr,low_ctr_visible_page,1108,1.5,0.0


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

### Weak picks + leakage check

The weakest pick in the top 20 is **content_0c8cb6bad4a7** because it has only 502 impressions, just above the 500-impression threshold. Its position is still strong at 2.0 and its CTR is 0.0%, so it remains a reasonable review candidate, but the smaller volume makes the signal less stable.

The rule does not use product flags, `trend_direction`, or `trend_pct`. It uses only 90-day performance fields and current page-level metrics. No future-window or label-derived inputs are used.

In [59]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
scoring_features = [
    "impressions_90d",
    "avg_position",
    "ctr"
]

forbidden_features = ["trend_direction", "trend_pct"]

print("Scoring features:")
print(scoring_features)

print("\nForbidden label-derived features used in scoring:")
used_forbidden = [col for col in forbidden_features if col in scoring_features]
print(used_forbidden)

print("\nProduct flag columns in dataset:")
print([
    col for col in df.columns
    if "flag" in col.lower() or "health_score" in col.lower()
])

assert len(used_forbidden) == 0

print("\nLeakage check passed.")

Scoring features:
['impressions_90d', 'avg_position', 'ctr']

Forbidden label-derived features used in scoring:
[]

Product flag columns in dataset:
[]

Leakage check passed.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.